# 🎙️ ZeroTTS - Google Colab (Headless / Chế Độ Trực Tiếp Không Cần WebUI)
[![GitHub Repo](https://img.shields.io/badge/GitHub-RevenantKitana%2FTSS-181717?logo=github)](https://github.com/RevenantKitana/TSS)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RevenantKitana/TSS/blob/main/ZeroTTS_Colab_Headless.ipynb)

> **Mô tả:** Bản Notebook chạy **trực tiếp trong Colab** mà không cần mở WebUI Server hay kết nối Cloudflare Tunnel.
> Hỗ trợ:
> - ⚡ Tổng hợp giọng nói nhanh, nghe thử tức thì trong Colab (`IPython.display.Audio`).
> - 📄 Xử lý kịch bản đơn lẻ hoặc hàng loạt (.docx / .txt / .md) theo chuẩn cú pháp tag `$[Thư_mục]`, `[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`.
> - 🎼 Tự động nối và xuất file `_FULL_MERGED.mp3/wav`.
> - 📁 Lưu trữ trực tiếp kết quả vào Google Drive (`MyDrive/ZeroTTS_Outputs`).
>
> **Lưu ý:** Trước khi chạy, hãy kiểm tra menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Nhấn **Save**.

In [ ]:
#@title Bước 1: Kết Nối Google Drive & Kiểm Tra Phần Cứng { run: "auto", display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Outputs" #@param {type:"string"}
DRIVE_INPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Inputs" #@param {type:"string"}

import os
import subprocess

print("🔍 1. Đang kiểm tra phần cứng GPU...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"   ✅ Phát hiện GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("   ℹ️ Đang chạy trên CPU. Để tăng tốc độ tối đa, bạn hãy vào: Runtime -> Change runtime type -> T4 GPU.")
    HAS_GPU = False

print("\n📁 2. Cấu hình thư mục lưu trữ:")
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        os.makedirs(DRIVE_INPUT_DIR, exist_ok=True)
        os.environ["ZEROTTS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
        os.environ["ZEROTTS_INPUT_DIR"] = DRIVE_INPUT_DIR
        print(f"   ✅ Google Drive đã kết nối thành công!")
        print(f"   📂 Thư mục xuất Audio:   {DRIVE_OUTPUT_DIR}")
        print(f"   📥 Thư mục lưu Kịch bản: {DRIVE_INPUT_DIR}")
    except Exception as e:
        print(f"   ⚠️ Không thể mount Google Drive ({e}). Sử dụng thư mục tạm /content/outputs và /content/ZeroTTS_Inputs.")
        os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
        os.environ["ZEROTTS_INPUT_DIR"] = "/content/ZeroTTS_Inputs"
        os.makedirs("/content/outputs", exist_ok=True)
        os.makedirs("/content/ZeroTTS_Inputs", exist_ok=True)
else:
    os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
    os.environ["ZEROTTS_INPUT_DIR"] = "/content/ZeroTTS_Inputs"
    os.makedirs("/content/outputs", exist_ok=True)
    os.makedirs("/content/ZeroTTS_Inputs", exist_ok=True)
    print("   📁 Dữ liệu sẽ lưu tạm tại /content/outputs (sẽ bị xoá khi tắt phiên Colab).")

In [ ]:
#@title Bước 2: Nạp Mã Nguồn Từ GitHub { display-mode: "form" }
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/RevenantKitana/TSS.git"
APP_DIR = "/content/TSS"

print(f"📥 Đang tải mã nguồn từ: {REPO_URL}...")
if not os.path.exists(os.path.join(APP_DIR, ".git")):
    shutil.rmtree(APP_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, APP_DIR], check=True)
    print("✅ Clone mã nguồn thành công từ RevenantKitana/TSS!")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    subprocess.run(["git", "-C", APP_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", APP_DIR, "reset", "--hard", "origin/main"], check=True)
    print("✅ Đã đồng bộ mã nguồn mới nhất từ RevenantKitana/TSS!")

# Tải Git LFS objects cho các gói giọng trong Voice_ZeroTTS_model
print("📦 Đang đồng bộ Git LFS (Voice embeddings & preview clips)...")
try:
    if shutil.which("git-lfs"):
        subprocess.run(["git", "lfs", "install"], check=False)
        subprocess.run(["git", "-C", APP_DIR, "lfs", "pull"], check=False)
except Exception:
    pass

# Đồng bộ thư mục webui, src và Voice_ZeroTTS_model
for folder in ["webui", "src", "Voice_ZeroTTS_model"]:
    src_f = os.path.join(APP_DIR, folder)
    dst_f = os.path.join("/content", folder)
    if os.path.exists(src_f):
        try:
            shutil.copytree(src_f, dst_f, dirs_exist_ok=True)
        except Exception:
            pass

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content", "/content/src", "/content/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

%cd $APP_DIR

In [ ]:
#@title Bước 3: Cài Đặt Thư Viện Hệ Thống & ONNX Runtime GPU { display-mode: "form" }
print("🔧 Đang cài đặt thư viện hệ thống và Python (chỉ mất ~1 phút)...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2 git-lfs

# 1. Cài đặt các gói phụ trợ
!pip install -q soundfile sounddevice fastapi uvicorn tokenizers huggingface_hub scipy requests pydantic
!pip install -q --no-deps -e .

# 2. Cài đặt ONNX Runtime GPU (Bản CUDA 12 tương thích hoàn hảo Colab T4)
if HAS_GPU:
    print("⚡ Đang cài đặt ONNX Runtime GPU (CUDA 12 T4)...\n")
    !pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null || true
    !pip install -q "onnxruntime-gpu>=1.19.0,<=1.20.1"
    !mkdir -p /etc/ld.so.conf.d/
    !find /usr/local/lib/python* /usr/lib/python* /usr/lib64-nvidia /usr/local/cuda* -name "lib" -type d -path "*nvidia*" 2>/dev/null > /etc/ld.so.conf.d/nvidia-libs.conf
    !echo "/usr/lib64-nvidia" >> /etc/ld.so.conf.d/nvidia-libs.conf
    !echo "/usr/local/cuda/lib64" >> /etc/ld.so.conf.d/nvidia-libs.conf
    !ldconfig 2>/dev/null || true
else:
    print("⚙️ Đang cài đặt ONNX Runtime CPU...")
    !pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null || true
    !pip install -q "onnxruntime>=1.17.0"

# 3. Kiểm tra và Preload CUDA
import os, sys
try:
    import torch
except Exception:
    pass

for mod in list(sys.modules.keys()):
    if "onnxruntime" in mod:
        del sys.modules[mod]
import onnxruntime as ort

if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception:
        pass
avail = ort.get_available_providers()
print(f"\n🔍 Kết quả kiểm tra ONNX Providers: {avail}")
if "CUDAExecutionProvider" in avail:
    print("🚀 [GPU T4] ĐÃ KÍCH HOẠT GPU THÀNH CÔNG! (CUDAExecutionProvider sẵn sàng)")
elif HAS_GPU:
    print("⚠️ Lưu ý: Nếu vừa cài đặt lại, hãy vào menu: Runtime -> Restart session rồi bấm Chạy lại Bước 3.")
else:
    print("ℹ️ Đang chạy trên CPU (CPUExecutionProvider).")

print("\n✅ Cài đặt hoàn tất! Môi trường đã sẵn sàng.")

In [ ]:
#@title Bước 4: Tải Model Weights & Khởi Tạo Bộ Giọng { display-mode: "form" }
from huggingface_hub import snapshot_download
import os
import sys
import subprocess

try:
    import torch
except Exception:
    pass

import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception:
        pass

avail_p = ort.get_available_providers()
print(f"🔍 ONNX Providers khả dụng: {avail_p}")
if "CUDAExecutionProvider" in avail_p:
    print("🚀 [GPU] CUDAExecutionProvider khả dụng!")
else:
    print("⚙️ [CPU] Đang dùng CPUExecutionProvider.")

APP_DIR = "/content/TSS" if os.path.exists("/content/TSS") else os.getcwd()
MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model") if os.path.exists(APP_DIR) else "/content/ZeroTTS_model"
os.environ["ZEROTTS_MODEL"] = MODEL_DIR

candidate_voices = [
    os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
    os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
    "/content/Voice_ZeroTTS_model/voices",
    "/content/Voice_ZeroTTS_model",
]
VOICES_DIR = next((p for p in candidate_voices if os.path.exists(p)), None)

print("📥 Đang tải weights mô hình ZeroTTS (~500MB)...")
snapshot_download(
    repo_id="zeroweight-ai/ZeroTTS",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False
)
print(f"✅ Đã tải xong Model Weights tại: {MODEL_DIR}")

if VOICES_DIR and os.path.exists(VOICES_DIR):
    voices = [f for f in os.listdir(VOICES_DIR) if not f.startswith(".") and os.path.isdir(os.path.join(VOICES_DIR, f))]
    print(f"🗣️ Đã xác thực {len(voices)} gói giọng sẵn có: {voices}")
else:
    print("ℹ️ Đang sử dụng gói giọng mặc định của mô hình.")

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content", "/content/src", "/content/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import webui.engine as engine
engine.set_model(MODEL_DIR, voices_dir=VOICES_DIR)
_ = engine.get_tts()
print(f"⚡ ZeroTTS Engine đã nạp sẵn sàng!")

In [ ]:
#@title Bước 5: Đọc Nhanh Văn Bản Trực Tiếp (Interactive TTS) { display-mode: "form" }
import time
import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display, HTML

#@markdown ### 📝 Nhập nội dung văn bản cần đọc:
TEXT_INPUT = "Chào mừng bạn đến với mô hình giọng đọc nhân tạo ZeroTTS Tiếng Việt chất lượng cao. [pause: 1.0s] Mô hình có khả năng tổng hợp giọng nói tự nhiên, truyền cảm và hỗ trợ điều chỉnh ngắt nghỉ linh hoạt!" #@param {type:"string"}

#@markdown ### 🎙️ Cấu hình giọng đọc & thông số:
VOICE_NAME = "maichi" #@param ["maichi", "baotrang", "kimoanh", "hamy", "giahuy", "huuduc", "quangminh", "tiendat", "unconditional"]
CFG_SCALE = 1.0 #@param {type:"slider", min:1.0, max:3.0, step:0.1}
AUDIO_TEMP = 0.8 #@param {type:"slider", min:0.3, max:1.2, step:0.05}
AUTO_PLAY = True #@param {type:"boolean"}
SAVE_FILE_NAME = "quick_tts_output.wav" #@param {type:"string"}

import webui.engine as engine

out_dir = os.environ.get("ZEROTTS_OUTPUT_DIR", "/content/outputs")
os.makedirs(out_dir, exist_ok=True)
out_file_path = os.path.join(out_dir, SAVE_FILE_NAME)

use_voice_flag = (VOICE_NAME != "unconditional")
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME

print(f"🔊 Đang tổng hợp giọng nói [{VOICE_NAME}]...")
start_t = time.time()

chunks = []
sample_rate = 24000

for sr, chunk in engine.generate_stream(
    text=TEXT_INPUT,
    voice_name=voice_param,
    cfg_scale=CFG_SCALE,
    audio_temperature=AUDIO_TEMP,
    use_voice=use_voice_flag,
):
    sample_rate = sr
    chunks.append(chunk)

if chunks:
    audio_data = np.concatenate(chunks)
    duration_sec = len(audio_data) / sample_rate
    elapsed_sec = time.time() - start_t
    rtf = elapsed_sec / max(duration_sec, 0.001)

    # Lưu file âm thanh
    sf.write(out_file_path, audio_data, sample_rate)
    
    print(f"✅ Hoàn thành trong {elapsed_sec:.2f}s | Thời lượng audio: {duration_sec:.2f}s | RTF: {rtf:.3f}")
    print(f"💾 File đã lưu tại: {out_file_path}")

    # Phát âm thanh trực tiếp trong Colab
    display(HTML(f"""
    <div style="background: #1e1e2e; border: 1px solid #7c3aed; padding: 15px; border-radius: 10px; margin: 10px 0; color: #f8fafc; font-family: sans-serif;">
        <div style="font-weight: bold; color: #a78bfa; margin-bottom: 8px;">🎙️ Nghe thử giọng đọc [{VOICE_NAME}]</div>
        <div style="font-size: 13px; color: #94a3b8; margin-bottom: 10px;">Thời lượng: {duration_sec:.2f}s | Tốc độ xử lý: {rtf:.3f}x Real-time</div>
    </div>
    """))
    display(Audio(audio_data, rate=sample_rate, autoplay=AUTO_PLAY))
else:
    print("⚠️ Không có dữ liệu audio nào được tạo ra.")

In [ ]:
#@title Bước 6: Tải Lên Kịch Bản (.docx / .txt / .md) Vào Google Drive { display-mode: "form" }
import os
import shutil
from google.colab import files

INPUT_DIR = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs")
if not os.path.exists(os.path.dirname(INPUT_DIR)) and not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/content/ZeroTTS_Inputs"
os.makedirs(INPUT_DIR, exist_ok=True)

print(f"📁 Thư mục kịch bản: {INPUT_DIR}")
print("📤 Chọn tệp kịch bản từ máy tính (.docx, .txt, .md) để nạp vào Google Drive:")

uploaded = files.upload()

LATEST_INPUT_FILE = None
if uploaded:
    for filename, content in uploaded.items():
        dest_path = os.path.join(INPUT_DIR, filename)
        with open(dest_path, "wb") as f:
            f.write(content)
        if os.path.exists(filename) and os.path.abspath(filename) != os.path.abspath(dest_path):
            try:
                os.remove(filename)
            except Exception:
                pass
        LATEST_INPUT_FILE = dest_path
        print(f"\n✅ Đã lưu vào Google Drive: {dest_path} ({len(content)/1024:.1f} KB)")

# Liệt kê kịch bản hiện có
print(f"\n📋 Danh sách kịch bản hiện có trong {INPUT_DIR}:")
found_files = []
for root, _, fnames in os.walk(INPUT_DIR):
    for fn in fnames:
        if fn.lower().endswith((".docx", ".txt", ".md")):
            fpath = os.path.join(root, fn)
            size_kb = os.path.getsize(fpath) / 1024
            mtime = os.path.getmtime(fpath)
            found_files.append((mtime, fpath, fn, size_kb))

if found_files:
    found_files.sort(key=lambda x: x[0], reverse=True)
    if not LATEST_INPUT_FILE:
        LATEST_INPUT_FILE = found_files[0][1]
    for i, (_, fp, fn, sz) in enumerate(found_files, 1):
        latest_mark = " 🌟 [Đang chọn]" if fp == LATEST_INPUT_FILE else ""
        print(f"  {i}. {fn} ({sz:.1f} KB){latest_mark}")
    print(f"\n👉 Bạn có thể xuống Bước 7 bấm Chạy ngay (sẽ tự động dùng tệp: {os.path.basename(LATEST_INPUT_FILE)})")
else:
    print("  (Chưa có tệp nào. Bạn hãy bấm Chạy ô này để tải file lên, hoặc dùng kịch bản mẫu ở Bước 7)")

In [ ]:
#@title Bước 7: Render Kịch Bản Dự Án Nâng Cao (Batch Tags & Multi-Project) { display-mode: "form" }
import os
import sys
import shutil
from IPython.display import Audio, display, HTML

#@markdown ### 📄 Chọn file kịch bản hoặc để trống để dùng mẫu bên dưới:
INPUT_FILE_PATH = "" #@param {type:"string"}

sample_script_text = """$[Bai_giang_01] // Tên thư mục dự án 1
[Cau_1]
Chào mừng các bạn đến với khóa đào tạo trực tuyến. [pause: 1.5s]

[Cau_2]
Hôm nay chúng ta sẽ tìm hiểu về các nguyên tắc cơ bản trong kỹ thuật âm thanh.

#[Ghi_chu_nhap]
Đoạn này là nháp hướng dẫn, hệ thống tự động bỏ qua không đọc.

[Cau_3]
Chúc các bạn có một buổi học thật hiệu quả!

$[Bai_giang_02] // Tên thư mục dự án 2
[Loi_chao]
Xin chào quý vị khán giả đã quay trở lại với chuyên mục công nghệ. [pause: 1.0s]
"""

#@markdown ### 🎙️ Cấu hình giọng đọc & Render:
VOICE_NAME = "maichi" #@param ["maichi", "baotrang", "kimoanh", "hamy", "giahuy", "huuduc", "quangminh", "tiendat", "unconditional"]
DEFAULT_PROJECT_NAME = "du_an_colab_batch" #@param {type:"string"}
CFG_SCALE = 1.0 #@param {type:"slider", min:1.0, max:3.0, step:0.1}
AUDIO_TEMP = 0.8 #@param {type:"slider", min:0.3, max:1.2, step:0.05}
AUTO_MERGE = True #@param {type:"boolean"}
MERGED_FORMAT = "MP3" #@param ["MP3", "WAV", "FLAC", "M4A", "OGG"]
NUM_WORKERS = 1 #@param [1, 2, 3, 4, 6, 8] {type:"raw"}
LISTEN_MERGED = True #@param {type:"boolean"}

APP_DIR = globals().get("APP_DIR") or "/content/TSS"
MODEL_DIR = globals().get("MODEL_DIR") or os.path.join(APP_DIR, "ZeroTTS_model")
VOICES_DIR = globals().get("VOICES_DIR") or None

if not VOICES_DIR:
    for vc in [
        os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
        os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
        "/content/TSS/Voice_ZeroTTS_model/voices",
        "/content/Voice_ZeroTTS_model/voices",
    ]:
        if os.path.exists(vc):
            VOICES_DIR = vc
            break

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content", "/content/src", "/content/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import webui.engine as engine
engine.set_model(MODEL_DIR, voices_dir=VOICES_DIR)

# Phát hiện tệp kịch bản
target_file = None
if INPUT_FILE_PATH.strip():
    cand = INPUT_FILE_PATH.strip()
    if os.path.exists(cand):
        target_file = cand
    else:
        in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs")
        cand_in_dir = os.path.join(in_dir, cand)
        if os.path.exists(cand_in_dir):
            target_file = cand_in_dir
        else:
            print(f"⚠️ Không tìm thấy '{cand}'. Sử dụng văn bản mẫu.")
elif "LATEST_INPUT_FILE" in globals() and LATEST_INPUT_FILE and os.path.exists(LATEST_INPUT_FILE):
    target_file = LATEST_INPUT_FILE
    print(f"💡 Đang dùng tệp kịch bản: {target_file}")
else:
    in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs")
    if os.path.exists(in_dir):
        files_in_dir = [os.path.join(in_dir, f) for f in os.listdir(in_dir) if f.lower().endswith((".docx", ".txt", ".md"))]
        if files_in_dir:
            files_in_dir.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            target_file = files_in_dir[0]
            print(f"💡 Tự động nạp kịch bản mới nhất từ Google Drive: {target_file}")

raw_script = sample_script_text
fallback_name = DEFAULT_PROJECT_NAME if "DEFAULT_PROJECT_NAME" in globals() else "du_an_colab_batch"
if target_file and os.path.exists(target_file):
    print(f"📄 Đang đọc dữ liệu từ tệp: {target_file}...")
    raw_script, default_f = engine.read_input_file(target_file)
    if default_f:
        fallback_name = default_f

projects = engine.parse_multi_project_blocks(raw_script, default_name=fallback_name)
print(f"🎯 Phân tích kịch bản: Nhận diện {len(projects)} dự án:")
for p in projects:
    print(f" 📁 Dự án: [{p['project_name']}] - Tổng số câu: {len(p['blocks'])}")
    for b in p['blocks']:
        st = "⏭️ BỎ QUA" if b['is_skipped'] else "✅ RENDER"
        print(f"    - [{b['tag']}]: {st} | {b['text'][:40]}...")

use_voice_flag = (VOICE_NAME != "unconditional")
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME

print(f"\n🔊 Bắt đầu render toàn bộ dự án với giọng [{VOICE_NAME}]...")
result_meta = {}
last_status = None
for status_msg, last_file, seg_text, q_meta in engine.generate_projects_queue_stream(
    text=raw_script,
    voice_name=voice_param,
    custom_name=fallback_name,
    auto_concat=AUTO_MERGE,
    merged_format=MERGED_FORMAT,
    cfg_scale=CFG_SCALE,
    audio_temperature=AUDIO_TEMP,
    use_voice=use_voice_flag,
    num_workers=int(NUM_WORKERS),
    result=result_meta,
):
    if status_msg and status_msg != last_status:
        last_status = status_msg
        print(f"   {status_msg}")

print(f"\n🎉 Hoàn thành toàn bộ kịch bản!")
all_folders = result_meta.get("folders", [])
merged_files_found = []
for out_f in all_folders:
    if os.path.exists(out_f):
        print(f"\n📂 Thư mục xuất: {out_f}")
        for f in sorted(os.listdir(out_f)):
            fpath = os.path.join(out_f, f)
            if "_FULL_MERGED" in f or "_merged" in f.lower():
                merged_files_found.append(fpath)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  ├── {f} ({size_kb:.1f} KB)")

# Phát file gộp trực tiếp
if LISTEN_MERGED and merged_files_found:
    for mf in merged_files_found:
        print(f"\n🎧 Phát âm thanh file nối hoàn chỉnh: {os.path.basename(mf)}")
        display(Audio(mf))